# Pandas vs Spark

## Différences entre Pandas et Spark

- **Pandas** est une bibliothèque Python populaire pour l'analyse et la manipulation de données. Elle fonctionne en mémoire et est idéale pour traiter des ensembles de données de petite à moyenne taille sur une seule machine.
- **Spark** (et PySpark) est un moteur de traitement distribué conçu pour le big data. Il permet de traiter de très grands volumes de données répartis sur plusieurs machines.

**Résumé :**
- Utilisez **Pandas** pour des analyses rapides sur des données qui tiennent en mémoire.
- Utilisez **Spark** pour traiter des données volumineuses ou distribuées, ou lorsque l'échelle dépasse les capacités d'une seule machine.

## Lire un dataset
- PySpark
- Pandas
- Pandas API sur Spark

Dans ce notebook il y a un paramètre `full_path` en haut du notebook

![](./Resources/NB13/img_parameters.png)

La cellule dessous permet de créer ce paramètre si il n'existe pas, il y aura une valeur par défaut vers l'emplacement du fichier MOCK_DATA.csv : `/Workspace/Users/{username}/databricks-training/Spark Developer/Python pour Data Science et Data Engineering/Resources/NB13/FINAL_DATA.csv`

Comme Spark ne peut pas lire de fichier directement dans le système de fichier du Workspace comme on l'as fait avec Pands, il faut : 
- Créer un volume (dans un Catalog/schema)
- Placer le fichier dans le volume

Les deux prochaines cellules vont paramètrer les différentes variables nécéssaires pour le `Setup`. 

Par défaut un volume sera créer dans le catalog : **workspace.default** (spark_training) mais on peut modifier les widgets (directement dans le code ou si la cellules à déjà été exécuté en haut du notebook).

In [0]:
# Passe la variable 'full_path' avec le chemin vers le fichier CSV MOCK_DATA.csv au notebook NB13/Setup
import os

full_path = os.getcwd() + "/Resources/NB13/FINAL_DATA.csv"

dbutils.widgets.text("full_path", full_path)
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume", "spark_training")

In [0]:
%run "./Resources/NB13/Setup"

### Lecture d'un dataset avec Spark

Pour lire un dataset avec PySpark, on utilise généralement la méthode `spark.read` adaptée au format du fichier (CSV, Parquet, etc.). Par exemple, pour lire un fichier CSV :

```python
df = spark.read.csv("/chemin/vers/le/fichier.csv", header=True, inferSchema=True)
```

- `spark.read.csv` : lit un fichier CSV.
- `header=True` : indique que la première ligne contient les noms de colonnes.
- `inferSchema=True` : permet à Spark de deviner le type des colonnes.


Pour lire une table d'un catalog avec PySpark, on utilise la méthode `spark.table` :

```python
df = spark.table("catalog.schema.nom_de_table")
```


- `spark.table` : lit une table existante dans le catalog Spark.
- Le nom de la table doit inclure le catalog et le schema si nécessaire.

Cela permet d'accéder directement aux données stockées dans le metastore, sans avoir à lire un fichier.

Le résultat est un DataFrame Spark, qui peut être manipulé avec des méthodes similaires à Pandas mais en mode distribué.

In [0]:
# les variables catalog, schema et volume sont définis dans le Setup (avec les valeurs des widgets)

df = spark.read.csv(
    f"/Volumes/{catalog}/{schema}/{volume}/FINAL_DATA.csv",
    header=True,
    inferSchema=True
)

display(df)

### Lecture d'un fichier CSV avec Pandas API sur Spark

Pour lire un fichier CSV avec Pandas API sur Spark, utilisez la méthode `ps.read_csv` :

```python
import pyspark.pandas as ps

df = ps.read_csv(:full_path)
```


- `ps.read_csv` : lit un fichier CSV en utilisant l'API Pandas sur Spark.
- `:full_path` : chemin du fichier CSV.

Le résultat est un DataFrame Pandas sur Spark, qui combine la syntaxe familière de Pandas avec la scalabilité de Spark.

In [0]:
import pyspark.pandas as ps 

df_ps = ps.read_csv(f"/Volumes/{catalog}/{schema}/{volume}/FINAL_DATA.csv", inferSchema=True, multiline=True)

display(df_ps)

### Les différents types d'index

- **sequence** :  
  Index séquentiel classique, similaire à celui de Pandas. Les valeurs d'index sont continues et ordonnées, généralement de 0 à n-1. Utilisé pour des DataFrames non distribués ou de petite taille.

- **distributed-sequence** :  
  Index séquentiel mais distribué sur plusieurs partitions. Chaque partition possède une séquence locale, ce qui permet de conserver l'ordre tout en profitant du traitement distribué. Utile pour les opérations nécessitant un ordre global.

- **distributed** :  
  Index distribué sans garantie d'ordre global. Les valeurs d'index sont réparties sur les partitions, mais ne suivent pas nécessairement une séquence continue. Optimisé pour le traitement parallèle et la scalabilité, mais l'ordre n'est pas garanti.

On peut modifier se paramètre avec l'option `compute.default_index_type`

In [0]:
ps.set_option("compute.default_index_type", "distributed-sequence")

df_dist_sequence = ps.read_csv(f"/Volumes/{catalog}/{schema}/{volume}/FINAL_DATA.csv", inferSchema=True, multiline=True)
df_dist_sequence.head()

### Conversion entre Spark DataFrame et Pandas API sur Spark

- **De Spark DataFrame vers Pandas API sur Spark :**

Utilisez la méthode `ps.DataFrame(df)` :
```python
df_converted = ps.DataFrame(df)
```

Utilisez la méthode `to_pandas_on_spark()` sur un Spark DataFrame si les cluster Databricks le permet:

```python
import pyspark.pandas as ps

# df_spark : Spark DataFrame
df_ps = df_spark.to_pandas_on_spark()
```


- **De Pandas API sur Spark vers Spark DataFrame :**

Utilisez la méthode `to_spark()` sur un DataFrame Pandas API sur Spark :

```python
# df_ps : DataFrame Pandas API sur Spark
df_spark = df_ps.to_spark()
```


Ces conversions permettent de profiter à la fois de la scalabilité de Spark et de la syntaxe familière de Pandas.

In [0]:
df_converted = ps.DataFrame(df)
display(df_converted)

In [0]:
df_spark = df_converted.to_spark()
display(df_spark)

### `count()` sur Spark et Pandas API

In [0]:
display(df_spark.groupBy("brand").count().orderBy("count", ascending=False))

In [0]:
df_converted["brand"].value_counts()

### Visualisation

In [0]:
df_converted["brand"].value_counts().hist(bins=20)

### SQL avec Pandas API sur Spark

On utilise la méthode `ps.sql(<QUERY>)`

In [0]:
# Register the pandas-on-Spark DataFrame as a temporary Spark view
df_converted.to_spark().createOrReplaceTempView("df_converted_view")

# Execute SQL query using Spark SQL
result = ps.sql("SELECT brand, COUNT(*) as count FROM df_converted_view GROUP BY brand ORDER BY count DESC")

display(result)